# Phase 3b — MMDetection Route B (native `mmdet` + RTMDet)

Rebuilds the detection stage on native **MMDetection** (RTMDet-s) instead of Ultralytics,
per the supervisor's ask, and integrates the enhancement/defogging transform as the
thesis's contribution.

Full background: `docs/MMDETECTION_ROUTE_B_RUNBOOK.md` (local-only, not pushed — see repo README).

**Run order:** mount Drive -> clone/pull repo -> §0.5 (download + preprocess BDD100K) ->
§Env setup (Cells 1-7, run once per session, does NOT survive a Colab disconnect) ->
Task 2 (YOLO->COCO) -> Task 3 (config sanity checks) -> Task 4 (train, costs compute) ->
Task 6 (with/without eval, costs compute).

**Runtime:** Colab, **T4 GPU**. Set this before running anything: `Runtime > Change runtime type > T4 GPU`.


## 0. Mount Drive and get the repo

Code comes from git. Datasets are downloaded fresh to local Colab disk in §0.5 below (not stored on Drive -- see §0.5 for why).

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
import os

REPO_DIR = '/content/computer_vision'
if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone https://github.com/Ib-Programmer/computer_vision.git {REPO_DIR}

%cd {REPO_DIR}


Cloning into '/content/computer_vision'...
remote: Enumerating objects: 995, done.
remote: Counting objects: 100% (203/203), done.
remote: Compressing objects: 100% (151/151), done.
remote: Total 995 (delta 130), reused 86 (delta 52), pack-reused 792 (from 1)
Receiving objects: 100% (995/995), 47.72 MiB | 18.89 MiB/s, done.
Resolving deltas: 100% (670/670), done.
/content/computer_vision


In [3]:
import shutil
import subprocess

nvidia_smi = shutil.which('nvidia-smi')
ok = False
if nvidia_smi:
    try:
        r = subprocess.run([nvidia_smi, '-L'], capture_output=True, text=True)
        ok = r.returncode == 0 and bool(r.stdout.strip())
        if ok:
            print(r.stdout)
    except OSError:
        ok = False

if not ok:
    raise RuntimeError(
        "No GPU visible to this runtime (nvidia-smi missing or reports no device). Go to "
        "Runtime > Change runtime type > T4 GPU, then Runtime > Restart session, then "
        "re-run from the top. (Everything below this cell will silently run on CPU "
        "otherwise -- slow, and gives meaningless latency numbers for the real-time claim.)"
    )


GPU 0: Tesla T4 (UUID: GPU-84005a97-98fa-75e3-73a8-9152b74db934)



### Browser keep-alive -- run once per session, right after the GPU check

Colab's idle-disconnect is driven by detected **browser tab** activity, not kernel
business -- it has already interrupted this notebook mid-cell twice (the `mm` env
creation and the torch install) with `CondaError: KeyboardInterrupt`, with no code
bug involved. Best-effort mitigation below; the reliable fallback is keeping the tab
focused/foregrounded for the long unattended stretches (env setup, Task 4 training).


In [4]:
from IPython.display import Javascript, display

display(Javascript(r'''
// Best-effort: Colab's frontend disconnects the runtime after a period with no detected
// UI activity in the tab -- independent of whether a cell is genuinely computing (e.g. a
// multi-GB download, or the mamba solve). That's what's been interrupting long setup/
// training cells with `CondaError: KeyboardInterrupt`, not a bug in the cell itself. This
// periodically clicks Colab's connect button to look like activity. It is a DOM hack
// against Colab's current UI and can silently go inert if Colab changes that markup --
// watch the browser console (F12) for repeating "[keepalive]" logs to confirm it is
// actually firing; if it stops finding a button, this stops helping and the reliable
// fallback is keeping the tab focused/foregrounded during long-running cells.
if (window.__phase3b_keepalive) { clearInterval(window.__phase3b_keepalive); }
window.__phase3b_keepalive = setInterval(() => {
  const btn = document.querySelector('colab-toolbar-button#connect') ||
              document.querySelector('#top-toolbar colab-connect-button') ||
              document.querySelector('paper-icon-button#connect');
  if (btn) { btn.click(); console.log('[keepalive] clicked', new Date().toISOString()); }
  else { console.log('[keepalive] no connect button found -- Colab UI may have changed, this is not helping'); }
}, 60000);
console.log('[keepalive] started');
'''))
print("Keep-alive JS injected for this browser tab. Open the browser console (F12) and "
      "confirm you see repeating '[keepalive] clicked' logs -- if you only see 'no connect "
      "button found', this mitigation isn't working and you should keep the tab focused "
      "manually during long cells instead.")


<IPython.core.display.Javascript object>

Keep-alive JS injected for this browser tab. Open the browser console (F12) and confirm you see repeating '[keepalive] clicked' logs -- if you only see 'no connect button found', this mitigation isn't working and you should keep the tab focused manually during long cells instead.


## 0.5 Download & preprocess BDD100K (needed by Task 2 below)

Task 2 (`yolo_to_coco.py`) reads `datasets/bdd100k_yolo/{train,val}/images|labels`.
Datasets are **not** persisted on Drive in this project -- every phase notebook
re-downloads to Colab's local SSD each session (see `Phase1_Data_Preparation.ipynb` /
`Phase3_Object_Detection.ipynb`; Drive is only used for results/checkpoints). This section
does the same, but only pulls BDD100K (~6.5GB, ~15-20 min) -- not the full 5-dataset set
Phase 1 downloads, since Route B only needs BDD100K.

Skips automatically if `datasets/bdd100k_yolo` already has labels from earlier in this
session (see the `[SKIP]` checks inside `preprocess_data.py`).


In [5]:
import os
from pathlib import Path

!pip install -q --upgrade kaggle gdown
try:
    from google.colab import userdata
    KAGGLE_API_TOKEN = userdata.get('KAGGLE_API_TOKEN')
except Exception:
    KAGGLE_API_TOKEN = None

if not KAGGLE_API_TOKEN:
    KAGGLE_API_TOKEN = 'KGAT_bbbc79ffbfa19a3fa2285815341158a2'

assert KAGGLE_API_TOKEN, 'No Kaggle token. Add KAGGLE_API_TOKEN to Colab Secrets or paste in cell.'

Path('/root/.kaggle').mkdir(parents=True, exist_ok=True)
token_file = Path('/root/.kaggle/access_token')
token_file.write_text(KAGGLE_API_TOKEN)
token_file.chmod(0o600)
os.environ['KAGGLE_API_TOKEN'] = KAGGLE_API_TOKEN

print('Verifying Kaggle auth...')
!kaggle datasets list -s "titanic" 2>&1 | head -3


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.2/126.2 kB 5.9 MB/s eta 0:00:00
Verifying Kaggle auth...
ref                                  title                                                size  lastUpdated                 downloadCount  voteCount  usabilityRating  
-----------------------------------  ---------------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
heptapod/titanic                     Titanic                                             11090  2017-05-16 08:14:22.210000         166126       2117  0.7058824        


In [6]:
%cd /content/computer_vision
!python scripts/download_datasets.py bdd100k
!python scripts/preprocess_data.py bdd100k


/content/computer_vision
Phase 1: Dataset Download
Base directory: /content/computer_vision
Datasets directory: /content/computer_vision/datasets

DOWNLOADING: BDD100K (Berkeley DeepDrive)
  Trying Kaggle CLI (solesensei/solesensei_bdd100k)...
  Kaggle CLI download successful (120000 images, 2 JSONs)
  Location: /content/computer_vision/datasets/bdd100k

DOWNLOAD SUMMARY
  bdd100k      -> 136000 images found

Done! Next: run preprocess_data.py
Phase 1: Data Preprocessing
Target size: (640, 640)
Split ratio: {'train': 0.7, 'val': 0.15, 'test': 0.15}
Chunk size: 200 images
JPEG quality: 90

PREPROCESSING: BDD100K → YOLO format
  Found train labels (consolidated): /content/computer_vision/datasets/bdd100k/bdd100k_labels_release/bdd100k/labels/bdd100k_labels_images_train.json
  Found val labels (consolidated): /content/computer_vision/datasets/bdd100k/bdd100k_labels_release/bdd100k/labels/bdd100k_labels_images_val.json
  Found train images: /content/computer_vision/datasets/bdd100k/bdd100k

### Recovery cell — run this any time paths/state look wrong

Every shell cell below already `cd`s into the repo itself before running anything, so this
isn't required for correctness — it's a fast standalone diagnostic. Useful after a Colab
disconnect/reconnect, or if you jumped into the middle of the notebook instead of running
top to bottom: tells you in a few seconds what still exists vs what needs re-running,
instead of guessing from a wall of errors further down.


In [7]:
import os
import subprocess

REPO_DIR = '/content/computer_vision'
MMDET_REPO = '/content/mmdetection'

get_ipython().run_line_magic('cd', REPO_DIR) if os.path.isdir(REPO_DIR) else print(f"[MISSING] {REPO_DIR} -- re-run the git clone/pull cell (§0).")

checks = [
    ("repo checked out", os.path.isdir(REPO_DIR)),
    ("BDD100K raw data downloaded (§0.5)", os.path.isdir(f"{REPO_DIR}/datasets/bdd100k")),
    ("BDD100K converted to YOLO layout (§0.5)", os.path.isdir(f"{REPO_DIR}/datasets/bdd100k_yolo")),
    ("mmdetection tools/ cloned (v3.3.0)", os.path.isdir(f"{MMDET_REPO}/tools")),
    ("mm conda env exists", os.path.isdir("/usr/local/envs/mm")),
    ("COCO annotations converted (Task 2)", os.path.exists(f"{REPO_DIR}/datasets/bdd100k_yolo/annotations/train.json")),
    ("Task 4 checkpoint exists", os.path.exists(f"{REPO_DIR}/work_dirs/rtmdet_bdd100k/latest.pth")),
]
for label, present in checks:
    print(f"[{'OK' if present else 'MISSING'}] {label}")

if os.path.isdir("/usr/local/envs/mm"):
    r = subprocess.run(['conda', 'run', '-n', 'mm', 'python', '-c',
                         "import torch; print('CUDA:', torch.cuda.is_available())"],
                        capture_output=True, text=True)
    print(r.stdout.strip() or r.stderr.strip())


/content/computer_vision
[OK] repo checked out
[OK] BDD100K raw data downloaded (§0.5)
[OK] BDD100K converted to YOLO layout (§0.5)
[MISSING] mmdetection tools/ cloned (v3.3.0)
[MISSING] mm conda env exists
[MISSING] COCO annotations converted (Task 2)
[MISSING] Task 4 checkpoint exists


In [8]:
!find /content/computer_vision/datasets/bdd100k -type d -name train | while read d; do echo "$(find "$d" -maxdepth 1 -type f | wc -l) $d"; done

7000 /content/computer_vision/datasets/bdd100k/bdd100k/bdd100k/images/10k/train
1156 /content/computer_vision/datasets/bdd100k/bdd100k/bdd100k/images/100k/train
7000 /content/computer_vision/datasets/bdd100k/bdd100k_seg/bdd100k/seg/images/train
7000 /content/computer_vision/datasets/bdd100k/bdd100k_seg/bdd100k/seg/labels/train
7000 /content/computer_vision/datasets/bdd100k/bdd100k_seg/bdd100k/seg/color_labels/train


In [9]:
!find /content/computer_vision/datasets/bdd100k -type f | sed 's|/[^/]*$||' | sort | uniq -c | sort -rn | head -20

  37216 /content/computer_vision/datasets/bdd100k/bdd100k/bdd100k/images/100k/train/trainA
  24750 /content/computer_vision/datasets/bdd100k/bdd100k/bdd100k/images/100k/train/trainB
  10632 /content/computer_vision/datasets/bdd100k/bdd100k/bdd100k/images/100k/test/trainA
  10000 /content/computer_vision/datasets/bdd100k/bdd100k/bdd100k/images/100k/val
   7109 /content/computer_vision/datasets/bdd100k/bdd100k/bdd100k/images/100k/test/trainB
   7000 /content/computer_vision/datasets/bdd100k/bdd100k_seg/bdd100k/seg/labels/train
   7000 /content/computer_vision/datasets/bdd100k/bdd100k_seg/bdd100k/seg/images/train
   7000 /content/computer_vision/datasets/bdd100k/bdd100k_seg/bdd100k/seg/color_labels/train
   7000 /content/computer_vision/datasets/bdd100k/bdd100k/bdd100k/images/10k/train
   4130 /content/computer_vision/datasets/bdd100k/bdd100k/bdd100k/images/100k/train/testA
   2748 /content/computer_vision/datasets/bdd100k/bdd100k/bdd100k/images/100k/train/testB
   2000 /content/computer_

In [10]:
!pip install -q opencv-python-headless tqdm pillow
!rm -rf /content/computer_vision/datasets/bdd100k_yolo
!cd /content/computer_vision && python scripts/preprocess_data.py bdd100k

Phase 1: Data Preprocessing
Target size: (640, 640)
Split ratio: {'train': 0.7, 'val': 0.15, 'test': 0.15}
Chunk size: 200 images
JPEG quality: 90

PREPROCESSING: BDD100K → YOLO format
  Found train labels (consolidated): /content/computer_vision/datasets/bdd100k/bdd100k_labels_release/bdd100k/labels/bdd100k_labels_images_train.json
  Found val labels (consolidated): /content/computer_vision/datasets/bdd100k/bdd100k_labels_release/bdd100k/labels/bdd100k_labels_images_val.json
  Found train images: /content/computer_vision/datasets/bdd100k/bdd100k/bdd100k/images/100k/train (1156 files)
  Found val images: /content/computer_vision/datasets/bdd100k/bdd100k/bdd100k/images/100k/val (10000 files)

  Loading train labels...
  69863 annotations loaded (consolidated)
    train: 100%|███████████████████████| 69863/69863 [00:01<00:00, 67152.35it/s]
    train: 1154 images converted, 68709 skipped, 1 backgrounds

  Loading val labels...
  10000 annotations loaded (consolidated)
    val: 100%|██████

In [11]:
p = '/content/computer_vision/scripts/preprocess_data.py'
s = open(p).read()

edits = [
    ("img_dirs = {}",
     "img_dirs = {}; img_index = {}"),
    ("img_dirs[split] = candidates[0]",
     "img_dirs[split] = candidates[0]; img_index[split] = {f.name: f for f in candidates[0].rglob('*.jpg')}"),
    ("img_count = len(list(candidates[0].glob('*.jpg')))",
     "img_count = len(img_index[split])"),
    ("img_path = img_dirs[split] / img_name",
     "img_path = img_index[split].get(img_name, img_dirs[split] / img_name)"),
]
for old, new in edits:
    assert s.count(old) == 1, f"expected exactly one match for: {old!r}, got {s.count(old)}"
    s = s.replace(old, new)

open(p, 'w').write(s)
print("patched")

patched


In [12]:
!rm -rf /content/computer_vision/datasets/bdd100k_yolo
!cd /content/computer_vision && python scripts/preprocess_data.py bdd100k

Phase 1: Data Preprocessing
Target size: (640, 640)
Split ratio: {'train': 0.7, 'val': 0.15, 'test': 0.15}
Chunk size: 200 images
JPEG quality: 90

PREPROCESSING: BDD100K → YOLO format
  Found train labels (consolidated): /content/computer_vision/datasets/bdd100k/bdd100k_labels_release/bdd100k/labels/bdd100k_labels_images_train.json
  Found val labels (consolidated): /content/computer_vision/datasets/bdd100k/bdd100k_labels_release/bdd100k/labels/bdd100k_labels_images_val.json
  Found train images: /content/computer_vision/datasets/bdd100k/bdd100k/bdd100k/images/100k/train (70000 files)
  Found val images: /content/computer_vision/datasets/bdd100k/bdd100k/bdd100k/images/100k/val (10000 files)

  Loading train labels...
  69863 annotations loaded (consolidated)
    train: 100%|████████████████████████| 69863/69863 [00:20<00:00, 3368.85it/s]
    train: 69863 images converted, 0 skipped, 14 backgrounds

  Loading val labels...
  10000 annotations loaded (consolidated)
    val: 100%|███████

In [13]:
if 'img_index' in s:
    print("already patched")
else:
    for old, new in edits:
        assert s.count(old) == 1, f"expected exactly one match for: {old!r}, got {s.count(old)}"
        s = s.replace(old, new)
    open(p, 'w').write(s)
    print("patched")

already patched


## 1. Environment setup (§3 of the runbook) — run once per session

Colab's default runtime (Python 3.12, torch 2.11, CUDA 12.8) is **incompatible** with the
OpenMMLab 2.x stack. `condacolab` gives us conda, then we build a separate **Python 3.10**
conda env (`mm`) with a pinned stack. The kernel itself stays 3.12 — every mmdet call below
routes through `conda run -n mm`.

Do not deviate from the pinned versions (torch 2.1.0+cu118 / mmcv 2.1.0 / mmdet 3.3.0 /
numpy<2) — see runbook §3 for why each pin exists.


In [14]:
# CELL 1 -- run ALONE. The kernel auto-restarts after this (expected). Do not re-run.
!pip install -q condacolab
import condacolab
import sys
condacolab.TARGET_PYTHON = '.'.join(map(str, sys.version_info[:2]))
condacolab.install()



📢 Announcement 📢
condacolab==0.2 will be released soon! Try it with:

    !pip install -q https://github.com/conda-incubator/condacolab/archive/main.zip
    import condacolab
    condacolab.install()

0.2.x introduces a new installation method based on Pixi, with customizable Python versions.
This may be breaking for your workflow. If that's the case, please report it at
https://github.com/conda-incubator/condacolab and pin your `pip install` command to
condacolab==0.1 as a workaround.

⏬ Downloading https://github.com/conda-forge/miniforge/releases/download/26.3.2-3/Miniforge3-26.3.2-3-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:10
🔁 Restarting kernel...


In [15]:
# Cell 1's condacolab.install() restarted the kernel, which resets the working directory
%cd /content/computer_vision


/content/computer_vision


In [16]:
# CELL 2 -- after the restart: create the 3.10 env (idempotent: safe to re-run after a
# disconnect/interrupt without manually removing anything first).
!conda run -n mm python -c "print(1)" > /dev/null 2>&1 && echo "[mm env already OK, skipping creation]" || { conda env remove -n mm -y > /dev/null 2>&1; mamba create -n mm python=3.10 -y -q || conda create -n mm python=3.10 -y -q; }


In [17]:
# CELL 2.5 -- (superseded, now a no-op) this used to force stdlib distutils
!conda env config vars unset -n mm SETUPTOOLS_USE_DISTUTILS > /dev/null 2>&1 || true
print("Cell 2.5 is now a no-op (env var unset if it was set from a prior session) -- see CELL 6.5 for the actual setuptools fix.")


Cell 2.5 is now a no-op (env var unset if it was set from a prior session) -- see CELL 6.5 for the actual setuptools fix.


In [18]:
# CELL 3 — GATE: must print 3.10.x before continuing. Stop here if it doesn't.
!conda run -n mm python -c "import sys; print('env Python:', sys.version.split()[0])"


env Python: 3.10.21


In [19]:
# CELL 4 — pinned torch (cu118 runs fine under the T4's 12.8 driver)
!conda run -n mm pip install -q torch==2.1.0 torchvision==0.16.0 torchaudio==2.1.0 \
    --index-url https://download.pytorch.org/whl/cu118
!conda run -n mm pip install -q "numpy<2"
!conda run -n mm python -c "import numpy,torch; print('numpy',numpy.__version__,'| torch',torch.__version__,'| CUDA',torch.cuda.is_available())"


numpy 1.26.4 | torch 2.1.0+cu118 | CUDA True


In [20]:
# CELL 5 — OpenMMLab stack (mmcv from the matching prebuilt index)
!conda run -n mm pip install -q -U openmim
!conda run -n mm mim install mmengine
!conda run -n mm mim install "mmcv==2.1.0" -f https://download.openmmlab.com/mmcv/dist/cu118/torch2.1/index.html
!conda run -n mm mim install "mmdet==3.3.0"


Looking in links: https://download.openmmlab.com/mmcv/dist/cu118/torch2.1.0/index.html
  Using cached numpy-2.2.6-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (62 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 MB 67.8 MB/s  0:00:01
Using cached numpy-2.2.6-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (16.8 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 135.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 122.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 74.4 MB/s  0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4

Looking in links: https://download.openmmlab.com/mmcv/dist/cu118/torch2.1/index.html, https://download.openmmlab.com/mmcv/dist/cu118/torch2.1.0/index.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.3/99.3 MB 6.7 MB/s  0:00:14

A module that was compiled usin

In [1]:
!conda run -n mm pip install -q "opencv-python<4.11" "numpy<2"
!conda run -n mm python -c "import numpy, cv2, torch; print('numpy', numpy.__version__, '| cv2', cv2.__version__, '| torch', torch.__version__)"


CondaError: KeyboardInterrupt

numpy 1.26.4 | cv2 5.0.0 | torch 2.1.0+cu118


In [2]:
# CELL 6.5 -- pin setuptools to a known-good version, AFTER Cells 4-6 (openmim/mmcv/mmdet

!conda run -n mm pip install -q --force-reinstall --no-deps "setuptools==69.5.1"
!conda run -n mm python -c "from torch.utils.cpp_extension import CUDA_HOME; import setuptools; print('setuptools', setuptools.__version__, '| CUDA_HOME import OK:', CUDA_HOME)"


setuptools 69.5.1 | CUDA_HOME import OK: /usr/local/cuda


In [3]:
%%bash
# CELL 6.6 -- fix HTTPS cert verification inside the mm env. Cell 6.5 fixed the
conda run -n mm pip install -q -U certifi
CERT_PATH=$(conda run -n mm python -c "import certifi; print(certifi.where())")
conda env config vars set -n mm SSL_CERT_FILE="$CERT_PATH" REQUESTS_CA_BUNDLE="$CERT_PATH"
# Real gate -- opens (but doesn't fully download) the exact checkpoint URL
# configs/rtmdet_bdd100k.py's load_from points at, the actual thing that was failing.
conda run -n mm python -c "
import urllib.request
url = 'https://download.openmmlab.com/mmdetection/v3.0/rtmdet/rtmdet_s_8xb32-300e_coco/rtmdet_s_8xb32-300e_coco_20220905_161602-387a891e.pth'
resp = urllib.request.urlopen(url, timeout=15)
print('HTTPS download OK -- status', resp.status)
resp.close()
"


HTTPS download OK -- status 200


In [4]:
# CELL 7 — smoke test. Success = "OK -- native MMDetection works."
!MPLBACKEND=Agg conda run -n mm python -c "import torch, mmcv, mmdet; \
print('mmdet', mmdet.__version__, '| CUDA', torch.cuda.is_available()); \
from mmdet.apis import DetInferencer; DetInferencer('rtmdet_tiny_8xb32-300e_coco'); \
print('OK -- native MMDetection works.')"


mmdet 3.3.0 | CUDA True
Loads checkpoint by http backend from path: https://download.openmmlab.com/mmdetection/v3.0/rtmdet/rtmdet_tiny_8xb32-300e_coco/rtmdet_tiny_8xb32-300e_coco_20220902_112414-78e30dcc.pth
The model and loaded state dict do not match exactly

unexpected key in source state_dict: data_preprocessor.mean, data_preprocessor.std

08/28 09:25:56 - mmengine - WARNING - Failed to search registry with scope "mmdet" in the "function" registry tree. As a workaround, the current "function" registry in "mmengine" is used to build instance. This may cause unexpected failure when running the built modules. Please check whether "mmdet" is a correct scope, or whether the registry is initialized.
OK -- native MMDetection works.
/usr/local/envs/mm/lib/python3.10/site-packages/mmengine/visualization/visualizer.py:196: UserWarning: Failed to add <class 'mmengine.visualization.vis_backend.LocalVisBackend'>, please provide the `save_dir` argument.
  warnings.warn(f'Failed to add {vis_backe

In [6]:
# `pip install mmdet` does NOT ship tools/train.py or tools/test.py as an importable
import os

MMDET_REPO = '/content/mmdetection'
if not os.path.isdir(MMDET_REPO):
    !git clone --depth 1 --branch v3.3.0 https://github.com/open-mmlab/mmdetection.git {MMDET_REPO}


### Optional — snapshot the env so you don't rebuild it every session

The `mm` env does **not** survive a Colab disconnect. Pack it once after Cell 7 passes,
then restore from the snapshot in future sessions instead of re-running Cells 1-6.


In [26]:
# Save (run once, after Cell 7 passes). Takes a while; ~2-4 GB on Drive.
!mkdir -p /content/drive/MyDrive/computer_vision
!pip install -q conda-pack
!conda pack -n mm -o /content/drive/MyDrive/computer_vision/mm_env.tar.gz --ignore-missing-files --force


usage: conda [-h] [-v] [--no-plugins] [-V] COMMAND ...
conda: error: argument COMMAND: invalid choice: 'pack' (choose from activate, check, clean, commands, compare, config, create, deactivate, doctor, env, export, info, init, install, list, menuinst, notices, package, remove, rename, repoquery, run, search, uninstall, update, upgrade)


In [27]:
import os

SNAPSHOT = '/content/drive/MyDrive/computer_vision/mm_env.tar.gz'
if not os.path.exists(SNAPSHOT):
    print(f"[WARN] no snapshot at {SNAPSHOT} yet -- run the save cell above first (once, "
          f"after Cell 7 passes), or just re-run Cells 1-6 this session instead of this cell.")
else:
    # Still need condacolab (Cell 1) first so /usr/local/envs exists as a conda-managed location.
    get_ipython().system('mkdir -p /usr/local/envs/mm')
    get_ipython().system(f'tar -xzf {SNAPSHOT} -C /usr/local/envs/mm')
    get_ipython().system("conda run -n mm python -c \"import torch, mmcv, mmdet; print('restored OK, mmdet', mmdet.__version__)\"")


restored OK, mmdet 3.3.0


## 2. Task 2 — YOLO -> COCO conversion

Reads `datasets/bdd100k_yolo/{train,val}/images|labels` (produced by §0.5 above, see
`scripts/preprocess_data.py`) and writes `datasets/bdd100k_yolo/annotations/{train,val}.json`.

Uses the class order that actually matches the on-disk labels — **not** alphabetical, see
`scripts/yolo_to_coco.py`'s header comment and runbook §1 for why this matters (a mismatch
here silently scrambles category ids with no error).


In [7]:
!cd /content/computer_vision && conda run -n mm python scripts/yolo_to_coco.py


YOLO -> COCO conversion (/content/computer_vision/datasets/bdd100k_yolo)
  train: converting 69863 images...
    train: 500/69863
    train: 1000/69863
    train: 1500/69863
    train: 2000/69863
    train: 2500/69863
    train: 3000/69863
    train: 3500/69863
    train: 4000/69863
    train: 4500/69863
    train: 5000/69863
    train: 5500/69863
    train: 6000/69863
    train: 6500/69863
    train: 7000/69863
    train: 7500/69863
    train: 8000/69863
    train: 8500/69863
    train: 9000/69863
    train: 9500/69863
    train: 10000/69863
    train: 10500/69863
    train: 11000/69863
    train: 11500/69863
    train: 12000/69863
    train: 12500/69863
    train: 13000/69863
    train: 13500/69863
    train: 14000/69863
    train: 14500/69863
    train: 15000/69863
    train: 15500/69863
    train: 16000/69863
    train: 16500/69863
    train: 17000/69863
    train: 17500/69863
    train: 18000/69863
    train: 18500/69863
    train: 19000/69863
    train: 19500/69863
    train: 200

In [8]:
%%bash
cd /content/computer_vision
conda run -n mm --no-capture-output python - << 'PY'
from pycocotools.coco import COCO
for split in ['train', 'val']:
    c = COCO(f'datasets/bdd100k_yolo/annotations/{split}.json')
    print(split, '-> images:', len(c.imgs), '| annotations:', len(c.anns), '| categories:', len(c.cats))
PY

loading annotations into memory...
Done (t=5.80s)
creating index...
index created!
train -> images: 69863 | annotations: 1185310 | categories: 10
loading annotations into memory...
Done (t=1.78s)
creating index...
index created!
val -> images: 10000 | annotations: 170805 | categories: 10


## 3. Task 3 — RTMDet config sanity checks

Before trusting `configs/rtmdet_bdd100k.py`, verify the two things flagged in its own
comments against the **installed** mmdet==3.3.0 (field paths and hook behavior have moved
between mmdet releases, so don't trust the skeleton blindly):

1. `bbox_head.num_classes` field path resolves correctly on the base RTMDet-s config.
2. The `PipelineSwitchHook` switch-epoch — the base config is tuned for 300 epochs; our
   fine-tune is 25 epochs, so the mosaic/mixup-off switch may never fire unless overridden.


In [9]:
%%bash
conda run -n mm --no-capture-output python - << 'PY'
import os, mmdet
from mmengine import Config
base = os.path.join(os.path.dirname(mmdet.__file__), '.mim', 'configs', 'rtmdet', 'rtmdet_s_8xb32-300e_coco.py')
c = Config.fromfile(base)
print('base bbox_head.num_classes:', c.model.bbox_head.num_classes)
for hook in c.custom_hooks:
    if 'PipelineSwitch' in hook.get('type', ''):
        print('PipelineSwitchHook switch_epoch:', hook.get('switch_epoch'))
PY

base bbox_head.num_classes: 80
PipelineSwitchHook switch_epoch: 280


In [10]:
!cat /content/computer_vision/configs/rtmdet_bdd100k.py

"""Route B / Task 3: RTMDet-s fine-tune on the BDD100K subset, native mmdet.

Run inside the conda `mm` env (see docs/MMDETECTION_ROUTE_B_RUNBOOK.md §3):
    !MPLBACKEND=Agg conda run -n mm python -m mmdet.tools.train configs/rtmdet_bdd100k.py

Verify this base config path and the bbox_head field path against the installed
mmdet==3.3.0 package before trusting them (they can move between releases):
    conda run -n mm python -c "from mmengine import Config; \
        c = Config.fromfile('mmdet::rtmdet/rtmdet_s_8xb32-300e_coco.py'); \
        print(c.model.bbox_head.num_classes)"
"""

_base_ = 'mmdet::rtmdet/rtmdet_s_8xb32-300e_coco.py'

data_root = 'datasets/bdd100k_yolo/'

# Order matches scripts/yolo_to_coco.py CLASSES, which matches the class_to_id used
# when the on-disk YOLO labels were generated (scripts/preprocess_data.py). This is
# NOT alphabetical and NOT the order in earlier runbook drafts — do not "fix" it.
classes = (
    'pedestrian', 'rider', 'car', 'truck', 'bus',
    't

In [11]:
%%writefile /content/computer_vision/configs/rtmdet_bdd100k.py
"""Route B / Task 3: RTMDet-s fine-tune on BDD100K, native mmdet.

Run inside the conda `mm` env (see docs/MMDETECTION_ROUTE_B_RUNBOOK.md §3):
    !cd /content/computer_vision && MPLBACKEND=Agg conda run -n mm python \
        /content/mmdetection/tools/train.py configs/rtmdet_bdd100k.py
"""

_base_ = 'mmdet::rtmdet/rtmdet_s_8xb32-300e_coco.py'

data_root = 'datasets/bdd100k_yolo/'

# Order matches scripts/yolo_to_coco.py CLASSES, which matches the class_to_id used
# when the on-disk YOLO labels were generated (scripts/preprocess_data.py). This is
# NOT alphabetical and NOT the order in earlier runbook drafts — do not "fix" it.
classes = (
    'pedestrian', 'rider', 'car', 'truck', 'bus',
    'train', 'motorcycle', 'bicycle', 'traffic light', 'traffic sign',
)
metainfo = dict(classes=classes)

model = dict(bbox_head=dict(num_classes=len(classes)))

# ---- Compute budget -------------------------------------------------------------
# Full BDD100K train is 69,863 images. At 25 epochs that is well beyond one Colab
# T4 session, so cap the train set. Set to None for the full split.
# Override at the CLI without editing: --cfg-options train_dataloader.dataset.indices=1
train_subset = 10000

train_dataloader = dict(
    batch_size=16,           # RTMDet-s at 640 fits a 16 GB T4 at 16; drop to 8 on OOM
    num_workers=2,
    dataset=dict(
        data_root=data_root,
        metainfo=metainfo,
        ann_file='annotations/train.json',
        data_prefix=dict(img='train/images/'),
        indices=train_subset,
    ),
)
val_dataloader = dict(
    dataset=dict(
        data_root=data_root,
        metainfo=metainfo,
        ann_file='annotations/val.json',
        data_prefix=dict(img='val/images/'),
    ),
)
test_dataloader = val_dataloader

val_evaluator = dict(ann_file=data_root + 'annotations/val.json')
test_evaluator = val_evaluator

# Base lr 0.004 is for 8 GPU x 32 = 256 images/step. Scale linearly to our batch.
auto_scale_lr = dict(enable=True, base_batch_size=256)

# Start from COCO-pretrained weights (transfer learning).
load_from = 'https://download.openmmlab.com/mmdetection/v3.0/rtmdet/rtmdet_s_8xb32-300e_coco/rtmdet_s_8xb32-300e_coco_20220905_161602-387a891e.pth'

# ---- Schedule -------------------------------------------------------------------
# The base config is tuned for 300 epochs and bakes those numbers in as literals:
# CosineAnnealingLR begin=150/end=300 and PipelineSwitchHook switch_epoch=280.
# Overriding max_epochs alone does NOT touch them (verified: cosine never ran and
# mosaic/mixup never switched off in a 25-epoch run). Redefine all of it here.
max_epochs = 25
train_cfg = dict(max_epochs=max_epochs, val_interval=5)

param_scheduler = [
    dict(type='LinearLR', start_factor=1e-5, by_epoch=False, begin=0, end=1000),
    dict(type='CosineAnnealingLR', eta_min=0.0002, begin=max_epochs // 2, end=max_epochs,
         T_max=max_epochs // 2, by_epoch=True, convert_to_iter_based=True),
]

# Must restate EMAHook: replacing custom_hooks replaces the whole list.
custom_hooks = [
    dict(type='EMAHook', ema_type='ExpMomentumEMA', momentum=0.0002,
         update_buffers=True, priority=49),
    dict(type='PipelineSwitchHook', switch_epoch=max_epochs - 5,
         switch_pipeline={{_base_.train_pipeline_stage2}}),
]

# Checkpoint on the same cadence as validation, keep the best by mAP.
default_hooks = dict(
    checkpoint=dict(interval=5, max_keep_ckpts=3, save_best='auto'),
)

Overwriting /content/computer_vision/configs/rtmdet_bdd100k.py


In [12]:
%%bash
cd /content/computer_vision
conda run -n mm --no-capture-output python - << 'PY'
from mmengine import Config
c = Config.fromfile('configs/rtmdet_bdd100k.py')
print('fine-tune bbox_head.num_classes:', c.model.bbox_head.num_classes)
print('max_epochs:', c.train_cfg.max_epochs, '| val_interval:', c.train_cfg.val_interval)
print('param_scheduler:', [(s['type'], s.get('begin'), s.get('end')) for s in c.param_scheduler])
print('switch_epoch:', [h.get('switch_epoch') for h in c.custom_hooks if 'PipelineSwitch' in h.get('type', '')])
print('checkpoint interval:', c.default_hooks.checkpoint.get('interval'))
PY

fine-tune bbox_head.num_classes: 10
max_epochs: 25 | val_interval: 5
param_scheduler: [('LinearLR', 0, 1000), ('CosineAnnealingLR', 12, 25)]
switch_epoch: [20]
checkpoint interval: 5


### 1-image overfit sanity check (cheap, ~1-2 min on T4)

Confirms the config, dataloader, and loss actually work end-to-end before committing a
full training run's compute budget. Loss should visibly decrease over a handful of iters.


In [13]:
!conda run -n mm pip install -q --force-reinstall --no-deps "setuptools==69.5.1"
!conda run -n mm python -c "import setuptools; print('setuptools', setuptools.__version__, '|', setuptools.__file__)"
!conda run -n mm python -c "from torch.utils.cpp_extension import CUDA_HOME; print('CUDA_HOME import OK:', CUDA_HOME)"


CondaError: KeyboardInterrupt

setuptools 69.5.1 | /usr/local/envs/mm/lib/python3.10/site-packages/setuptools/__init__.py
CUDA_HOME import OK: /usr/local/cuda


In [14]:
!cd /content/computer_vision && MPLBACKEND=Agg conda run -n mm --no-capture-output python /content/mmdetection/tools/train.py configs/rtmdet_bdd100k.py \
    --work-dir work_dirs/overfit_check \
    --cfg-options train_dataloader.dataset.indices=1 train_cfg.max_epochs=30 train_cfg.val_interval=100 \
                  param_scheduler.0.end=5 default_hooks.logger.interval=1 default_hooks.checkpoint.interval=100

08/28 09:28:34 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.10.21 | packaged by conda-forge | (main, Aug 21 2026, 22:42:31) [GCC 14.4.0]
    CUDA available: True
    MUSA available: False
    numpy_random_seed: 1477089315
    GPU 0: Tesla T4
    CUDA_HOME: /usr/local/cuda
    NVCC: Cuda compilation tools, release 12.8, V12.8.93
    GCC: gcc (Ubuntu 11.4.0-1ubuntu1~22.04.3) 11.4.0
    PyTorch: 2.1.0+cu118
    PyTorch compiling details: PyTorch built with:
  - GCC 9.3
  - C++ Version: 201703
  - Intel(R) oneAPI Math Kernel Library Version 2022.2-Product Build 20220804 for Intel(R) 64 architecture applications
  - Intel(R) MKL-DNN v3.1.1 (Git Hash 64f6bcbcbab628e96f33a62c3e975f8535a7bde4)
  - OpenMP 201511 (a.k.a. OpenMP 4.5)
  - LAPACK is enabled (usually provided by MKL)
  - NNPACK is enabled
  - CPU capability usage: AVX512
  - CUDA Runtime 11.8
  - NVCC architecture flags: -gencode;arch=comp

In [15]:
!cd /content/computer_vision && MPLBACKEND=Agg conda run -n mm --no-capture-output python /content/mmdetection/tools/train.py configs/rtmdet_bdd100k.py \
    --work-dir work_dirs/overfit_check \
    --cfg-options train_dataloader.dataset.indices=1 val_dataloader.dataset.indices=50 \
                  train_cfg.max_epochs=30 train_cfg.val_interval=100 \
                  custom_hooks.1.switch_epoch=1 param_scheduler.0.end=5 \
                  auto_scale_lr.enable=False optim_wrapper.optimizer.lr=0.001 \
                  default_hooks.logger.interval=1 default_hooks.checkpoint.interval=100

08/28 09:49:10 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.10.21 | packaged by conda-forge | (main, Aug 21 2026, 22:42:31) [GCC 14.4.0]
    CUDA available: True
    MUSA available: False
    numpy_random_seed: 783419616
    GPU 0: Tesla T4
    CUDA_HOME: /usr/local/cuda
    NVCC: Cuda compilation tools, release 12.8, V12.8.93
    GCC: gcc (Ubuntu 11.4.0-1ubuntu1~22.04.3) 11.4.0
    PyTorch: 2.1.0+cu118
    PyTorch compiling details: PyTorch built with:
  - GCC 9.3
  - C++ Version: 201703
  - Intel(R) oneAPI Math Kernel Library Version 2022.2-Product Build 20220804 for Intel(R) 64 architecture applications
  - Intel(R) MKL-DNN v3.1.1 (Git Hash 64f6bcbcbab628e96f33a62c3e975f8535a7bde4)
  - OpenMP 201511 (a.k.a. OpenMP 4.5)
  - LAPACK is enabled (usually provided by MKL)
  - NNPACK is enabled
  - CPU capability usage: AVX512
  - CUDA Runtime 11.8
  - NVCC architecture flags: -gencode;arch=compu

In [16]:
!grep "Epoch(train)" /content/computer_vision/work_dirs/overfit_check/*/*.log | tail -n 30 | awk '{for(i=1;i<=NF;i++) if($i=="Epoch(train)"||$i=="loss:"||$i=="loss_cls:"||$i=="loss_bbox:") printf "%s %s ", $i, $(i+1); print ""}'

Epoch(train) [1][1/1] loss: 2.7408 loss_cls: 2.0831 loss_bbox: 0.6576 
Epoch(train) [2][1/1] loss: 2.6323 loss_cls: 1.9098 loss_bbox: 0.7225 
Epoch(train) [3][1/1] loss: 2.7348 loss_cls: 2.0597 loss_bbox: 0.6752 
Epoch(train) [4][1/1] loss: 2.7504 loss_cls: 2.0794 loss_bbox: 0.6710 
Epoch(train) [5][1/1] loss: 2.8186 loss_cls: 2.1895 loss_bbox: 0.6291 
Epoch(train) [6][1/1] loss: 2.8159 loss_cls: 2.1991 loss_bbox: 0.6168 
Epoch(train) [7][1/1] loss: 2.8169 loss_cls: 2.2231 loss_bbox: 0.5939 
Epoch(train) [8][1/1] loss: 2.8086 loss_cls: 2.2301 loss_bbox: 0.5785 
Epoch(train) [9][1/1] loss: 2.7691 loss_cls: 2.2007 loss_bbox: 0.5684 
Epoch(train) [10][1/1] loss: 2.7382 loss_cls: 2.1574 loss_bbox: 0.5808 
Epoch(train) [11][1/1] loss: 2.6889 loss_cls: 2.1295 loss_bbox: 0.5594 
Epoch(train) [12][1/1] loss: 2.6081 loss_cls: 2.0680 loss_bbox: 0.5401 
Epoch(train) [13][1/1] loss: 2.5110 loss_cls: 1.9938 loss_bbox: 0.5172 
Epoch(train) [14][1/1] loss: 2.4427 loss_cls: 1.9277 loss_bbox: 0.5150 
E

In [17]:
%%writefile /content/computer_vision/configs/rtmdet_bdd100k.py
"""Route B / Task 3: RTMDet-s fine-tune on BDD100K, native mmdet.

Run inside the conda `mm` env (see docs/MMDETECTION_ROUTE_B_RUNBOOK.md §3):
    !cd /content/computer_vision && MPLBACKEND=Agg conda run -n mm python \
        /content/mmdetection/tools/train.py configs/rtmdet_bdd100k.py
"""

_base_ = 'mmdet::rtmdet/rtmdet_s_8xb32-300e_coco.py'

data_root = 'datasets/bdd100k_yolo/'

# Order matches scripts/yolo_to_coco.py CLASSES, which matches the class_to_id used
# when the on-disk YOLO labels were generated (scripts/preprocess_data.py). This is
# NOT alphabetical and NOT the order in earlier runbook drafts — do not "fix" it.
classes = (
    'pedestrian', 'rider', 'car', 'truck', 'bus',
    'train', 'motorcycle', 'bicycle', 'traffic light', 'traffic sign',
)
metainfo = dict(classes=classes)

model = dict(bbox_head=dict(num_classes=len(classes)))

# ---- Compute budget -------------------------------------------------------------
# Full BDD100K train is 69,863 images. At 25 epochs that is well beyond one Colab
# T4 session, so cap the train set. Set to None for the full split.
# Override at the CLI without editing: --cfg-options train_dataloader.dataset.indices=1
train_subset = 10000

train_dataloader = dict(
    batch_size=16,           # RTMDet-s at 640 fits a 16 GB T4 at 16; drop to 8 on OOM
    num_workers=2,
    dataset=dict(
        data_root=data_root,
        metainfo=metainfo,
        ann_file='annotations/train.json',
        data_prefix=dict(img='train/images/'),
        indices=train_subset,
    ),
)
val_dataloader = dict(
    num_workers=2,           # base config uses 10; Colab has 2 CPUs
    dataset=dict(
        data_root=data_root,
        metainfo=metainfo,
        ann_file='annotations/val.json',
        data_prefix=dict(img='val/images/'),
    ),
)
test_dataloader = val_dataloader

val_evaluator = dict(ann_file=data_root + 'annotations/val.json')
test_evaluator = val_evaluator

# Base lr 0.004 is for 8 GPU x 32 = 256 images/step. Scale linearly to our batch.
auto_scale_lr = dict(enable=True, base_batch_size=256)

# Start from COCO-pretrained weights (transfer learning).
load_from = 'https://download.openmmlab.com/mmdetection/v3.0/rtmdet/rtmdet_s_8xb32-300e_coco/rtmdet_s_8xb32-300e_coco_20220905_161602-387a891e.pth'

# ---- Schedule -------------------------------------------------------------------
# The base config is tuned for 300 epochs and bakes those numbers in as literals:
# CosineAnnealingLR begin=150/end=300 and PipelineSwitchHook switch_epoch=280.
# Overriding max_epochs alone does NOT touch them (verified: cosine never ran and
# mosaic/mixup never switched off in a 25-epoch run). Redefine all of it here.
max_epochs = 25
train_cfg = dict(max_epochs=max_epochs, val_interval=5)

param_scheduler = [
    dict(type='LinearLR', start_factor=1e-5, by_epoch=False, begin=0, end=1000),
    dict(type='CosineAnnealingLR', eta_min=0.0002, begin=max_epochs // 2, end=max_epochs,
         T_max=max_epochs // 2, by_epoch=True, convert_to_iter_based=True),
]

# Must restate EMAHook: replacing custom_hooks replaces the whole list.
custom_hooks = [
    dict(type='EMAHook', ema_type='ExpMomentumEMA', momentum=0.0002,
         update_buffers=True, priority=49),
    dict(type='PipelineSwitchHook', switch_epoch=max_epochs - 5,
         switch_pipeline={{_base_.train_pipeline_stage2}}),
]

# Checkpoint on the same cadence as validation, keep the best by mAP.
default_hooks = dict(
    checkpoint=dict(interval=5, max_keep_ckpts=3, save_best='auto'),
)

Overwriting /content/computer_vision/configs/rtmdet_bdd100k.py


In [18]:
%%bash
cd /content/computer_vision
conda run -n mm --no-capture-output python - << 'PY'
from mmengine import Config
c = Config.fromfile('configs/rtmdet_bdd100k.py')
print('num_classes:', c.model.bbox_head.num_classes, '| max_epochs:', c.train_cfg.max_epochs, '| val_interval:', c.train_cfg.val_interval)
print('param_scheduler:', [(s['type'], s.get('begin'), s.get('end')) for s in c.param_scheduler])
print('switch_epoch:', [h.get('switch_epoch') for h in c.custom_hooks if 'PipelineSwitch' in h.get('type', '')])
print('checkpoint interval:', c.default_hooks.checkpoint.get('interval'), '| val num_workers:', c.val_dataloader.num_workers)
PY

num_classes: 10 | max_epochs: 25 | val_interval: 5
param_scheduler: [('LinearLR', 0, 1000), ('CosineAnnealingLR', 12, 25)]
switch_epoch: [20]
checkpoint interval: 5 | val num_workers: 2


## 4. Task 4 — baseline train + eval (costs real compute — budget check before running)

Reproduces the Phase 3 baseline inside mmdet. Expect low absolute mAP given the small
subset — that's fine, this is the baseline the enhancement comparison (Task 6) is measured
against, not a production number.


In [ ]:
!cd /content/computer_vision && MPLBACKEND=Agg conda run -n mm --no-capture-output python /content/mmdetection/tools/train.py configs/rtmdet_bdd100k.py

08/28 09:57:39 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.10.21 | packaged by conda-forge | (main, Aug 21 2026, 22:42:31) [GCC 14.4.0]
    CUDA available: True
    MUSA available: False
    numpy_random_seed: 860664979
    GPU 0: Tesla T4
    CUDA_HOME: /usr/local/cuda
    NVCC: Cuda compilation tools, release 12.8, V12.8.93
    GCC: gcc (Ubuntu 11.4.0-1ubuntu1~22.04.3) 11.4.0
    PyTorch: 2.1.0+cu118
    PyTorch compiling details: PyTorch built with:
  - GCC 9.3
  - C++ Version: 201703
  - Intel(R) oneAPI Math Kernel Library Version 2022.2-Product Build 20220804 for Intel(R) 64 architecture applications
  - Intel(R) MKL-DNN v3.1.1 (Git Hash 64f6bcbcbab628e96f33a62c3e975f8535a7bde4)
  - OpenMP 201511 (a.k.a. OpenMP 4.5)
  - LAPACK is enabled (usually provided by MKL)
  - NNPACK is enabled
  - CPU capability usage: AVX512
  - CUDA Runtime 11.8
  - NVCC architecture flags: -gencode;arch=compu

In [1]:
from google.colab import drive; drive.mount('/content/drive')
!ls -la /content/drive/MyDrive/computer_vision/ckpt/ 2>/dev/null; \
 ls -la /content/drive/MyDrive/computer_vision/work_dirs/rtmdet_bdd100k/*.pth 2>/dev/null

Mounted at /content/drive


## 5. Task 6 — with/without enhancement evaluation (the thesis result)

Runs eval twice — `EnhanceImage` off vs on (`method='zero_dce'`, the resolved real-time
path) — across available conditions, and reports COCO mAP + measured per-frame enhancement
latency (`results['enhance_latency_ms']`) against the ~25-30 FPS end-to-end target.

Wire `EnhanceImage` into `test_pipeline` (see `scripts/mm_transforms.py` docstring for the
exact insertion point — right after `LoadImageFromFile`) before running the "with
enhancement" pass. Keep a copy of the config without it for the "without" baseline pass.


In [ ]:
# Baseline (no enhancement) — uses configs/rtmdet_bdd100k.py + the checkpoint from Task 4.
!cd /content/computer_vision && MPLBACKEND=Agg conda run -n mm python /content/mmdetection/tools/test.py configs/rtmdet_bdd100k.py \
    work_dirs/rtmdet_bdd100k/latest.pth --out results_baseline.pkl


In [ ]:
# With enhancement — point at a config variant that adds EnhanceImage to test_pipeline
!cd /content/computer_vision && PYTHONPATH=/content/computer_vision MPLBACKEND=Agg conda run -n mm python /content/mmdetection/tools/test.py configs/rtmdet_bdd100k_enhanced.py \
    work_dirs/rtmdet_bdd100k/latest.pth --out results_enhanced.pkl
